In [1]:
import numpy as np
import pandas as pd

def off_policy_eval(df, policy, seed=0):
    """
    df: DataFrame с колонками:
        - action: залогированный action (0/1)
        - reward: 0/1 (клик)
    policy: "random" или "first_action"
    """
    rng = np.random.default_rng(seed)

    a_logged = df["action"].to_numpy().astype(int)
    r = df["reward"].to_numpy().astype(float)
    n = len(df)

    # logging policy: uniform
    mu = 0.5

    # target policy
    if policy == "random":
        # тестируемая политика: равномерный рандом
        a_target = rng.integers(0, 2, size=n)
        # вероятность выбрать именно залогированный action при target policy
        pi_logged = np.full(n, 0.5)

    elif policy == "first_action":
        # тестируемая политика: всегда выбирает action=0 (первый)
        a_target = np.zeros(n, dtype=int)
        pi_logged = (a_logged == 0).astype(float)  # 1 если залогированный action=0, иначе 0

    else:
        raise ValueError("policy must be 'random' or 'first_action'")

    # пересечения (для "наивной" оценки по совпадениям)
    match = (a_target == a_logged)
    n_intersections = int(match.sum())

    sum_reward = float(r[match].sum())
    ctr = (sum_reward / n_intersections) if n_intersections > 0 else np.nan

    # IPS
    w = pi_logged / mu              # = pi(a_logged|x) / mu(a_logged|x)
    sum_ips_reward = float((w * r).sum())
    ips_ctr = sum_ips_reward / n    # стандартная IPS-оценка E[r]

    # (опционально) self-normalized IPS:
    # snips_ctr = sum_ips_reward / w.sum() if w.sum() > 0 else np.nan

    return {
        "policy": policy,
        "n": n,
        "n_intersections": n_intersections,
        "ctr": ctr,
        "ips_ctr": ips_ctr,
        "sum_reward": sum_reward,
        "sum_ips_reward": sum_ips_reward,
    }

# ---- пример использования ----
# df должен быть вашим логом:
# df = pd.read_csv("log.csv")  # columns: action, reward

# пример синтетики:
df = pd.DataFrame({
    "action": np.random.randint(0, 2, size=10000),   # логгирующая равномерная
    "reward": (np.random.rand(10000) < 0.1).astype(int)  # клики с вероятностью 0.1
})

print(off_policy_eval(df, "random", seed=42))
print(off_policy_eval(df, "first_action", seed=42))


{'policy': 'random', 'n': 10000, 'n_intersections': 5041, 'ctr': 0.09839317595715136, 'ips_ctr': 0.0953, 'sum_reward': 496.0, 'sum_ips_reward': 953.0}
{'policy': 'first_action', 'n': 10000, 'n_intersections': 4996, 'ctr': 0.09327461969575661, 'ips_ctr': 0.0932, 'sum_reward': 466.0, 'sum_ips_reward': 932.0}


In [ ]:
496.0

In [2]:
10000 * 0.09839317595715136

983.9317595715136